# 02 · Explicaciones LIME y su evaluación cuantitativa (versión corregida)

Reemplaza a `Lime_.ipynb`. Requiere haber ejecutado `01_entrenamiento.ipynb`.

**Principales correcciones**

| | Original | Corregido |
|---|---|---|
| Imágenes explicadas | carpeta `images_or` (incluye imágenes de entrenamiento) | solo imágenes del **test**, muestra aleatoria estratificada o todas |
| Modelo | redefinido a mano en el notebook | `pl.load_trained_model()`: la misma función que en el entrenamiento |
| Entrada a LIME | imagen ya normalizada, sin recorte central | imagen RGB [0,1] con el mismo redimensionado; se normaliza dentro de `predict` |
| `hide_color=0` | gris medio de ImageNet | negro, como dice el artículo |
| Semilla | explicador sin `random_state` | `random_state` fijo |
| Quick Shift | juicio visual sobre 1 imagen | R² local y estabilidad top-k sobre varias imágenes de validación |
| `batch_size` de LIME | tratado como hiperparámetro | solo parámetro de cómputo (no cambia la explicación) |
| Patrones | lectura cualitativa | fragmentación, confianza, solapamiento con regiones anotadas, prueba de aleatorización y Grad-CAM |

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pipeline as pl

pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Configuración

In [ ]:
cfg = pl.Config(
    originals_dir='C:/Users/Usuario/Desktop/images_or',
    labels_csv='C:/Users/Usuario/Desktop/etiquetas_originales.csv',
    output_dir='resultados',
)
EXP, SEED = 8, cfg.seeds[0]          # modelo a explicar (EfficientNet V2-M, Exp. 8)
ARCH = pl.ARCH_OF_EXP[EXP]
N_PER_CLASS = None                   # None = TODAS las imágenes del test; o un número (muestra aleatoria)
MASKS_DIR = 'mascaras'               # mascaras/<categoria>/<imagen>.png (ver sección 6)
LIME_DIR = os.path.join(cfg.output_dir, 'lime')
os.makedirs(LIME_DIR, exist_ok=True)

In [ ]:
model = pl.load_trained_model(EXP, SEED, cfg)
predict_fn = pl.make_predict_fn(model, cfg.device)

manifest = pd.read_csv(os.path.join(cfg.output_dir, 'manifiestos', f'split_seed{SEED}.csv'))
manifest['path'] = manifest['image'].map(lambda n: os.path.join(cfg.originals_dir, n))
test_df = manifest[manifest.split == 'test'].reset_index(drop=True)
val_df = manifest[manifest.split == 'val'].reset_index(drop=True)
print(len(test_df), 'imágenes de test;', len(val_df), 'de validación')

## 2. Muestra a explicar  [FIX-23]
Sin selección manual: todo el test o una muestra aleatoria estratificada por clase **verdadera** con semilla fija.

In [ ]:
if N_PER_CLASS is None:
    lime_df = test_df.copy()
else:
    lime_df = (test_df.groupby('label', group_keys=False)
               .apply(lambda g: g.sample(min(len(g), N_PER_CLASS), random_state=SEED))
               .reset_index(drop=True))
lime_df['label'].map(dict(enumerate(pl.CLASS_NAMES))).value_counts()

## 3. Selección cuantitativa de Quick Shift sobre validación  [FIX-22]
Rejilla reducida por costo; amplíela si hay tiempo de GPU. Criterio: número razonable de segmentos, luego estabilidad del top-5 entre semillas y R² local.

In [ ]:
rng = np.random.default_rng(SEED)
qs_images = [pl.load_image_01(p, ARCH, cfg)
             for p in val_df['path'].sample(10, random_state=SEED)]
qs = pl.select_quickshift(qs_images, predict_fn,
                          kernel_sizes=[2, 4, 6, 8], max_dists=[100, 300, 500],
                          ratios=[0.1, 0.3, 0.6], num_samples=500, seeds=(0, 1))
qs.to_csv(os.path.join(LIME_DIR, 'seleccion_quickshift.csv'), index=False)
QS = qs.iloc[0][['kernel_size', 'max_dist', 'ratio']].to_dict()
QS = {'kernel_size': int(QS['kernel_size']), 'max_dist': int(QS['max_dist']), 'ratio': float(QS['ratio'])}
print('Quick Shift elegido:', QS)
qs.head(10)

## 4. Número de muestras de LIME según estabilidad  [FIX-20]
`batch_size` ya no se evalúa: solo define cuántas perturbaciones se procesan a la vez y no cambia la explicación.

In [ ]:
rows = []
for ns in [1000, 3000, 5000]:
    jac, r2 = [], []
    for img in qs_images:
        res = [pl.explain(img, predict_fn, **QS, num_samples=ns, random_state=s) for s in (0, 1, 2)]
        jac += [pl.jaccard_topk(res[0]['weights'], r['weights'], 5) for r in res[1:]]
        r2 += [r['score'] for r in res]
    rows.append({'num_samples': ns, 'jaccard_top5': np.nanmean(jac), 'r2': np.mean(r2)})
ns_table = pd.DataFrame(rows)
ns_table.to_csv(os.path.join(LIME_DIR, 'estabilidad_num_samples.csv'), index=False)
stable = ns_table[ns_table.jaccard_top5 >= 0.8]
NUM_SAMPLES = int(stable.num_samples.min() if len(stable) else ns_table.num_samples.max())
print('num_samples elegido:', NUM_SAMPLES)
ns_table

## 5. LIME sobre la muestra y cuantificación de patrones  [R1, R4]

In [ ]:
explanations, rows = {}, []
for _, r in lime_df.iterrows():
    img = pl.load_image_01(r['path'], ARCH, cfg)
    res = pl.explain(img, predict_fn, **QS, num_samples=NUM_SAMPLES, random_state=SEED)
    explanations[r['image']] = res
    np.savez_compressed(os.path.join(LIME_DIR, f"{os.path.splitext(r['image'])[0]}.npz"),
                        segments=res['segments'],
                        weight_ids=np.array(list(res['weights'].keys())),
                        weight_values=np.array(list(res['weights'].values())))
    rows.append({'image': r['image'], 'true': pl.CLASS_NAMES[r['label']],
                 'pred': pl.CLASS_NAMES[res['label']], 'correct': r['label'] == res['label'],
                 'confidence': res['prob'], 'lime_r2': res['score'], 'n_segments': res['n_segments'],
                 **pl.fragmentation(res['weights'])})
lime_table = pd.DataFrame(rows)
lime_table.to_csv(os.path.join(LIME_DIR, 'lime_por_imagen.csv'), index=False)
lime_table.groupby('pred')[['confidence', 'lime_r2', 'entropy_norm', 'n50', 'n50_frac']].agg(['mean', 'std', 'count'])

In [ ]:
# ¿La 'fragmentación' es propia de la clase Neutral o de la baja confianza?  (R4.4b)
groups = [g['entropy_norm'].dropna() for _, g in lime_table.groupby('pred')]
groups = [g for g in groups if len(g) >= 2]
if len(groups) >= 2:
    print('Kruskal-Wallis entropía ~ clase predicha:', stats.kruskal(*groups))
else:
    print('Kruskal-Wallis no aplicable: menos de dos clases predichas con ≥2 imágenes.')
print('Spearman entropía vs confianza:', stats.spearmanr(lime_table['entropy_norm'],
                                                          lime_table['confidence'], nan_policy='omit'))

# Regresión entropía ~ clase + confianza, con IC95% bootstrap
X = (pd.get_dummies(lime_table['pred']).reindex(columns=pl.CLASS_NAMES, fill_value=0)
     [['Neutral', 'Negative']].astype(float))   # Positive = categoría de referencia
X['confidence'] = lime_table['confidence']
X.insert(0, 'intercept', 1.0)
data = pd.concat([X, lime_table['entropy_norm']], axis=1).dropna()
data = data.loc[:, (data != 0).any()]  # quita clases sin imágenes predichas
coef = lambda d: np.linalg.lstsq(d.iloc[:, :-1].values, d.iloc[:, -1].values, rcond=None)[0]
boot = np.array([coef(data.sample(len(data), replace=True, random_state=i)) for i in range(1000)])
pd.DataFrame({'coef': coef(data), 'ci_low': np.percentile(boot, 2.5, 0),
              'ci_high': np.percentile(boot, 97.5, 0)}, index=data.columns[:-1])

## 6. Solapamiento con regiones anotadas por humanos  [R1: "validation" requiere ground truth]
Estructura esperada: `mascaras/<categoria>/<nombre_imagen>.png`, máscara binaria (blanco = región) del mismo tamaño que la imagen original. Categorías sugeridas: `producto`, `cara`, `cuerpo`, `texto`. Se compara con un baseline de superpíxeles al azar.

In [ ]:
CATEGORIES = ['producto', 'cara', 'cuerpo', 'texto']
if os.path.isdir(MASKS_DIR):
    rows = []
    for name, res in explanations.items():
        stem = os.path.splitext(name)[0]
        for cat in CATEGORIES:
            mpath = os.path.join(MASKS_DIR, cat, stem + '.png')
            if os.path.exists(mpath):
                ov = pl.region_overlap(res, pl.load_mask(mpath, ARCH, cfg), k=3)
                if ov:
                    rows.append({'image': name, 'category': cat,
                                 'pred': pl.CLASS_NAMES[res['label']], **ov})
    overlap = pd.DataFrame(rows)
    overlap.to_csv(os.path.join(LIME_DIR, 'solapamiento_regiones.csv'), index=False)
    if overlap.empty:
        print('No hay máscaras que coincidan con las imágenes explicadas (o sin pesos LIME positivos).')
    else:
      display(overlap.groupby(['pred', 'category'])[['hit_top1', 'hit_random', 'precision_topk',
                                                      'precision_random', 'iou_topk', 'iou_random']]
              .mean().assign(n=overlap.groupby(['pred', 'category']).size()))
else:
    print(f'No existe {MASKS_DIR}/: anote las regiones para cuantificar los patrones de producto, '
          'cara, cuerpo y texto.')

## 7. Prueba de aleatorización de parámetros (Adebayo et al., 2018)  [R1]
Si las explicaciones dependen de lo aprendido, deben parecerse cada vez menos a las originales al reinicializar capas de la cabeza.

In [ ]:
sanity_imgs = list(explanations)[:20]
rows = []
for n in range(1, pl.n_head_layers(model) + 1):
    rnd_fn = pl.make_predict_fn(pl.randomized_copy(model, n, seed=SEED), cfg.device)
    for name in sanity_imgs:
        path = lime_df.loc[lime_df.image == name, 'path'].iloc[0]
        img = pl.load_image_01(path, ARCH, cfg)
        res_r = pl.explain(img, rnd_fn, **QS, num_samples=NUM_SAMPLES, random_state=SEED)
        rows.append({'layers_randomized': n, 'image': name,
                     'spearman': pl.spearman_between(explanations[name], res_r)})
sanity = pd.DataFrame(rows)
sanity.to_csv(os.path.join(LIME_DIR, 'prueba_aleatorizacion.csv'), index=False)
sanity.groupby('layers_randomized')['spearman'].describe()

## 8. Comparación con Grad-CAM  [R1]

In [ ]:
rows, cams = [], {}
for name, res in explanations.items():
    path = lime_df.loc[lime_df.image == name, 'path'].iloc[0]
    img = pl.load_image_01(path, ARCH, cfg)
    cam = pl.gradcam(model, img, res['label'], cfg.device)
    cams[name] = cam
    top_cam = pl.segment_means(cam, res['segments'])
    cam_w = {i: top_cam[i] for i in res['weights']}
    rows.append({'image': name, 'pred': pl.CLASS_NAMES[res['label']],
                 'spearman_lime_gradcam': pl.spearman_lime_vs_map(res, cam),
                 'jaccard_top5': pl.jaccard_topk(res['weights'], cam_w, 5)})
agreement = pd.DataFrame(rows)
agreement.to_csv(os.path.join(LIME_DIR, 'lime_vs_gradcam.csv'), index=False)
agreement.groupby('pred')[['spearman_lime_gradcam', 'jaccard_top5']].agg(['mean', 'std', 'count'])

## 9. Figuras de ejemplo  [FIX-17, FIX-21]
Títulos con las clases correctas y barra de color etiquetada como peso LIME (no probabilidad).

In [ ]:
def plot_example(name, ax_row):
    res = explanations[name]
    path = lime_df.loc[lime_df.image == name, 'path'].iloc[0]
    img = pl.load_image_01(path, ARCH, cfg)
    wm = pl.weight_map(res)
    lim = np.abs(wm).max()
    true = lime_df.loc[lime_df.image == name, 'label'].iloc[0]
    ax_row[0].imshow(img)
    ax_row[0].set_title(f'True: {pl.CLASS_NAMES[true]} | Pred: {pl.CLASS_NAMES[res["label"]]} ({res["prob"]:.2f})')
    im = ax_row[1].imshow(wm, cmap='RdBu', vmin=-lim, vmax=lim)
    ax_row[1].set_title('LIME')
    plt.colorbar(im, ax=ax_row[1], fraction=0.046, label='LIME weight')
    ax_row[2].imshow(img)
    ax_row[2].imshow(cams[name], cmap='jet', alpha=0.45)
    ax_row[2].set_title('Grad-CAM')
    for a in ax_row:
        a.axis('off')

examples = lime_table.sort_values('confidence', ascending=False).groupby('pred').head(1)['image'].tolist()
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4 * len(examples)), squeeze=False)
for name, row in zip(examples, axes):
    plot_example(name, row)
plt.tight_layout()
plt.savefig(os.path.join(LIME_DIR, 'ejemplos.png'), dpi=200)
plt.show()